# SVHN Digit Classification: SVM Baseline vs Deep CNNs

This project builds an image classification pipeline for the Street View House Numbers (SVHN) dataset. The goal is to classify 32×32 colour digit images into classes 0–9.

The notebook compares three approaches:

1. A traditional Support Vector Machine (SVM) baseline
2. A Deep Convolutional Neural Network (DCNN) trained on raw images
3. A DCNN with image data augmentation for improved generalisation

The project demonstrates practical computer vision workflow skills: data loading, preprocessing, baseline modelling, CNN design, data augmentation, model evaluation, error analysis, and model export.


## Project Overview

Street View House Numbers is a real-world digit recognition dataset captured from natural scenes. Compared with clean handwritten digit datasets, SVHN images can include background clutter, lighting changes, partial crops, and multiple visual distractions.

This notebook approaches the task as a practical computer vision project: start with a simple baseline, train deeper image models, then compare performance using accuracy, F1 score, learning curves, confusion matrices, and incorrect prediction examples.


## 1. Imports and Environment Setup

In [ ]:
import os
from pathlib import Path
from time import process_time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.io import loadmat

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.utils import to_categorical

from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
print("TensorFlow version:", tf.__version__)

## 2. Locate Dataset Files

This helper searches common Kaggle and local notebook paths so the notebook can run after the dataset is attached through the Kaggle sidebar.

In [ ]:
def find_file(filename, roots=(".", "/kaggle/input", "/mnt/data")):
    """Search common Kaggle/local locations for a file."""
    for root in roots:
        root_path = Path(root)
        if not root_path.exists():
            continue
        matches = list(root_path.rglob(filename))
        # Ignore macOS metadata folders if present
        matches = [m for m in matches if "__MACOSX" not in str(m)]
        if matches:
            return matches[0]
    raise FileNotFoundError(f"Could not find {filename}. Add the dataset files to the notebook.")

def find_first_available(filenames):
    for filename in filenames:
        try:
            return find_file(filename)
        except FileNotFoundError:
            continue
    raise FileNotFoundError(f"Could not find any of these files: {filenames}")

train_path = find_first_available(["svhn_train.mat", "q3_train.mat"])
val_path = find_first_available(["svhn_validation.mat", "q3_validation.mat"])
test_path = find_first_available(["svhn_test.mat", "q3_test.mat"])

print("Train:", train_path)
print("Validation:", val_path)
print("Test:", test_path)

## 3. Load and Preprocess Data

In [ ]:
def load_data(train_path, val_path, test_path):
    train = loadmat(train_path)
    val = loadmat(val_path)
    test = loadmat(test_path)

    train_X = train["train_X"].astype("float32") / 255.0
    train_y = train["train_y"].reshape(-1).astype("int64")

    val_X = val["val_X"].astype("float32") / 255.0
    val_y = val["val_y"].reshape(-1).astype("int64")

    test_X = test["test_X"].astype("float32") / 255.0
    test_y = test["test_y"].reshape(-1).astype("int64")

    return train_X, train_y, val_X, val_y, test_X, test_y

train_X, train_y, val_X, val_y, test_X, test_y = load_data(train_path, val_path, test_path)

print("Training images:", train_X.shape)
print("Validation images:", val_X.shape)
print("Testing images:", test_X.shape)
print("Classes:", np.unique(train_y))

train_y_cat = to_categorical(train_y, 10)
val_y_cat = to_categorical(val_y, 10)
test_y_cat = to_categorical(test_y, 10)

## 4. Explore Sample Images

In [ ]:
def plot_images(images, labels, n=20):
    plt.figure(figsize=(10, 5))
    for i in range(min(n, len(images))):
        plt.subplot(4, 5, i + 1)
        plt.imshow(images[i])
        plt.title(f"Label: {labels[i]}")
        plt.axis("off")
    plt.tight_layout()
    plt.show()

plot_images(train_X, train_y)

## 5. Baseline Model: Support Vector Machine

The SVM uses flattened image pixels to create a simple traditional machine learning benchmark. This provides a useful baseline before moving to convolutional models that can learn spatial image patterns.

In [ ]:
def vectorise(images):
    return images.reshape((images.shape[0], -1))

train_vector_X = vectorise(train_X)
val_vector_X = vectorise(val_X)
test_vector_X = vectorise(test_X)

svm_train_start = process_time()
svm = SVC(kernel="linear", C=1.0)
svm.fit(train_vector_X, train_y)
svm_train_time = process_time() - svm_train_start

svm_infer_start = process_time()
svm_predictions = svm.predict(test_vector_X)
svm_inference_time = process_time() - svm_infer_start

svm_accuracy = accuracy_score(test_y, svm_predictions)
svm_f1 = f1_score(test_y, svm_predictions, average="macro")

print(f"SVM accuracy: {svm_accuracy:.4f}")
print(f"SVM macro F1: {svm_f1:.4f}")
print(f"SVM training time: {svm_train_time:.2f}s")
print(f"SVM inference time: {svm_inference_time:.2f}s")

## 6. Deep CNN Without Data Augmentation

This model uses convolutional layers to learn spatial features directly from the 32×32 colour images.

In [ ]:
def build_dcnn(input_shape=(32, 32, 3), num_classes=10):
    inputs = keras.Input(shape=input_shape, name="input_image")

    x = layers.Conv2D(32, (3, 3), activation="relu")(inputs)
    x = layers.MaxPooling2D((2, 2))(x)

    x = layers.Conv2D(64, (3, 3), activation="relu")(x)
    x = layers.MaxPooling2D((2, 2))(x)

    x = layers.Conv2D(128, (3, 3), activation="relu")(x)
    x = layers.Flatten()(x)

    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.5)(x)

    outputs = layers.Dense(num_classes, activation="softmax")(x)

    model = keras.Model(inputs=inputs, outputs=outputs, name="svhn_dcnn_no_augmentation")
    model.compile(
        optimizer="adam",
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model

model = build_dcnn()
model.summary()

In [ ]:
early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
)

start_train = process_time()
history = model.fit(
    train_X,
    train_y_cat,
    epochs=30,
    batch_size=64,
    validation_data=(val_X, val_y_cat),
    callbacks=[early_stop],
    verbose=1,
)
train_time = process_time() - start_train

start_test = process_time()
dcnn_predictions_proba = model.predict(test_X, verbose=0)
test_inference_time = process_time() - start_test

dcnn_predictions = np.argmax(dcnn_predictions_proba, axis=1)
dcnn_accuracy = accuracy_score(test_y, dcnn_predictions)
dcnn_f1 = f1_score(test_y, dcnn_predictions, average="macro")

print(f"DCNN accuracy: {dcnn_accuracy:.4f}")
print(f"DCNN macro F1: {dcnn_f1:.4f}")
print(f"DCNN training time: {train_time:.2f}s")
print(f"DCNN inference time: {test_inference_time:.2f}s")

## 7. Deep CNN With Data Augmentation

Data augmentation is applied inside the model using Keras preprocessing layers. This makes the training process more robust to small translations, rotations, and zoom variations commonly seen in real-world digit images.

In [ ]:
def build_augmented_dcnn(input_shape=(32, 32, 3), num_classes=10):
    data_augmentation = keras.Sequential(
        [
            layers.RandomRotation(0.05),
            layers.RandomZoom(0.025),
            layers.RandomTranslation(height_factor=0.025, width_factor=0.025),
        ],
        name="data_augmentation",
    )

    inputs = keras.Input(shape=input_shape, name="input_image")
    x = data_augmentation(inputs)

    x = layers.Conv2D(32, (3, 3), activation="relu")(x)
    x = layers.MaxPooling2D((2, 2))(x)

    x = layers.Conv2D(64, (3, 3), activation="relu")(x)
    x = layers.MaxPooling2D((2, 2))(x)

    x = layers.Conv2D(128, (3, 3), activation="relu")(x)
    x = layers.Flatten()(x)

    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.5)(x)

    outputs = layers.Dense(num_classes, activation="softmax")(x)

    model = keras.Model(inputs=inputs, outputs=outputs, name="svhn_dcnn_with_augmentation")
    model.compile(
        optimizer="adam",
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model

aug_model = build_augmented_dcnn()
aug_model.summary()

In [ ]:
start_aug_train = process_time()
aug_history = aug_model.fit(
    train_X,
    train_y_cat,
    epochs=30,
    batch_size=64,
    validation_data=(val_X, val_y_cat),
    callbacks=[early_stop],
    verbose=1,
)
aug_train_time = process_time() - start_aug_train

start_aug_test = process_time()
aug_predictions_proba = aug_model.predict(test_X, verbose=0)
aug_inference_time = process_time() - start_aug_test

aug_predictions = np.argmax(aug_predictions_proba, axis=1)
aug_accuracy = accuracy_score(test_y, aug_predictions)
aug_f1 = f1_score(test_y, aug_predictions, average="macro")

print(f"Augmented DCNN accuracy: {aug_accuracy:.4f}")
print(f"Augmented DCNN macro F1: {aug_f1:.4f}")
print(f"Augmented DCNN training time: {aug_train_time:.2f}s")
print(f"Augmented DCNN inference time: {aug_inference_time:.2f}s")

## 8. Evaluation Functions

In [ ]:
def plot_learning_curves(history_obj, title):
    history_dict = history_obj.history

    plt.figure(figsize=(8, 5))
    plt.plot(history_dict["loss"], label="Training loss")
    plt.plot(history_dict["val_loss"], label="Validation loss")
    plt.plot(history_dict["accuracy"], label="Training accuracy")
    plt.plot(history_dict["val_accuracy"], label="Validation accuracy")
    plt.title(title)
    plt.xlabel("Epoch")
    plt.ylabel("Value")
    plt.legend()
    plt.grid(True)
    plt.show()

plot_learning_curves(history, "DCNN without Augmentation")
plot_learning_curves(aug_history, "DCNN with Augmentation")

In [ ]:
print("Classification report: DCNN with augmentation")
print(classification_report(test_y, aug_predictions))

cm = confusion_matrix(test_y, aug_predictions)
fig, ax = plt.subplots(figsize=(8, 8))
ConfusionMatrixDisplay(cm, display_labels=range(10)).plot(ax=ax, cmap="Blues", colorbar=False)
plt.title("Confusion Matrix - Augmented DCNN")
plt.show()

## 9. Model Comparison

In [ ]:
results = pd.DataFrame(
    [
        {
            "Model": "SVM",
            "Accuracy": svm_accuracy,
            "Macro F1": svm_f1,
            "Training Time (s)": svm_train_time,
            "Inference Time (s)": svm_inference_time,
        },
        {
            "Model": "DCNN without augmentation",
            "Accuracy": dcnn_accuracy,
            "Macro F1": dcnn_f1,
            "Training Time (s)": train_time,
            "Inference Time (s)": test_inference_time,
        },
        {
            "Model": "DCNN with augmentation",
            "Accuracy": aug_accuracy,
            "Macro F1": aug_f1,
            "Training Time (s)": aug_train_time,
            "Inference Time (s)": aug_inference_time,
        },
    ]
)

results

## 10. Failure Case Analysis

The grid below shows examples where the augmented CNN made an incorrect prediction. This helps identify whether errors come from blur, ambiguous digits, cropping, lighting, or background noise.

In [ ]:
def plot_failure_cases(images, true_labels, predicted_labels, n_cases=20):
    failure_indices = np.where(predicted_labels != true_labels)[0]

    if len(failure_indices) == 0:
        print("No failure cases found.")
        return

    selected = failure_indices[:n_cases]
    plt.figure(figsize=(10, 5))

    for i, idx in enumerate(selected):
        plt.subplot(4, 5, i + 1)
        plt.imshow(images[idx])
        plt.title(f"T:{true_labels[idx]} P:{predicted_labels[idx]}")
        plt.axis("off")

    plt.tight_layout()
    plt.show()

plot_failure_cases(test_X, test_y, aug_predictions)

## 11. Export the Trained Model

Kaggle saves outputs to `/kaggle/working`. After running the notebook, the trained model can be downloaded from the notebook output files.

In [ ]:
output_path = "/kaggle/working/svhn_augmented_dcnn_model.keras" if Path("/kaggle/working").exists() else "svhn_augmented_dcnn_model.keras"
aug_model.save(output_path)
print("Saved model to:", output_path)

## Conclusion

The SVM provides a simple traditional machine learning baseline, but it treats images as flat feature vectors. The DCNN improves performance by learning spatial features from the images. The augmented DCNN adds small random image transformations during training, which can improve robustness and reduce overfitting when training data is limited.

This project demonstrates image classification, deep learning model design, data augmentation, model evaluation, and result interpretation using TensorFlow/Keras.